In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATASET_PATH = r'G:\AI\Plant_Disease_Detection_System\model\Dataset\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)'
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VALID_DIR = os.path.join(DATASET_PATH, 'valid')

print("Train classes found:", len(os.listdir(TRAIN_DIR)))
print("Validation classes found:", len(os.listdir(VALID_DIR)))


Train classes found: 38
Validation classes found: 38


In [2]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# --- FIX 1: Heavy Data Augmentation with Color Variations ---
# This forces the model to learn textures instead of just color spots
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3], # Simulates varying lighting conditions from Google images
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)


Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.


In [3]:
# Load base model
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), 
    include_top=False, 
    weights='imagenet'
)

# --- FIX 2: Fine-Tuning the Base Model ---
# We unfreeze the top layers of MobileNetV2 so it can learn plant-specific details (like spots and fuzz)
base_model.trainable = True
# Let's freeze the first 100 layers and unfreeze the rest
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Build Model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(), # Stabilizes learning during fine-tuning
    layers.Dropout(0.4),        # Increased to prevent overfitting
    layers.Dense(38, activation='softmax') 
])

# --- FIX 3: Lower Learning Rate for Fine-Tuning ---
# A lower learning rate prevents destroying the pre-trained ImageNet weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("🧠 Architecture Built & Fine-Tuning Enabled Successfully!")

# --- FIX 4: Extended Patience ---
# Fine-tuning takes a bit more time to balance out, so we increase patience to 3
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

print("🚀 Commencing Model Training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8, # Bumped slightly since early stopping will save us if it plateaus
    callbacks=[early_stopping]
)

model.save('plant_disease_model.h5')
print("🎉 Process Complete! 'plant_disease_model.h5' has been saved.")

🧠 Architecture Built & Fine-Tuning Enabled Successfully!
🚀 Commencing Model Training...
Epoch 1/8
  11/2197 ━━━━━━━━━━━━━━━━━━━━ 1:34:21 3s/step - accuracy: 0.0284 - loss: 4.6708

KeyboardInterrupt: 